# Penalty cap — optimal bid and success rate

A model of how a solver responds to the **penalty cap** $c_l$. A solver bids $s$ and wins the
auction iff $s \ge s_{\text{ref}}$, where the competing solvers' reference bid $s_{\text{ref}}$ is
**uniform on $[0, 200]$**. On a win it realises a routing
**surplus** $\sigma$ — a random draw of how good its execution turns out. Conditional on winning:

- **reward** (paid on delivery): $R = \min(c_u,\, s - s_{\text{ref}})$
- **penalty** (paid on unset): $P = \min(c_l,\, s_{\text{ref}})$
- **deliver payoff** $R + \sigma - s$,  **unset payoff** $-P$.

**Two kinds of unset.**

- **Forced** — with probability $p_{\text{fail}}$ the routing fails outright (an infrastructure /
  submission failure) and the solver must unset and pay the penalty. This is the **base unset
  rate**: a fixed floor, independent of $\sigma$ and the caps.
- **Strategic** — among the non-forced draws the solver delivers only when the surplus clears a
  pre-planned threshold, otherwise it unsets rather than deliver at too large a loss:

  $$\text{deliver} \iff \sigma \ge \bar\sigma(s), \qquad
    \bar\sigma(s) = \mathbb{E}\big[\sigma^*(s, s_{\text{ref}}) \mid \text{win}\big], \qquad
    \sigma^* = s - R - P.$$

  This is the **routing-specific unset rate**, and it grows with the token's **volatility** — the
  spread $\sigma_{\text{std}}$ of the surplus distribution. A volatile token throws more low /
  negative draws below $\bar\sigma(s)$, so the solver unsets more. The threshold is fixed before
  $s_{\text{ref}}$ is revealed, so it uses only the win-conditional *expectation* of the reference,
  never its realised value.

The surplus is drawn from a Gaussian $\mathcal{N}(\mu, \sigma_{\text{std}})$ whose support includes
negative values, so a wide (volatile) distribution genuinely produces loss-making draws. Naming conventions:
 $c_l \leftrightarrow$ `penalty_cap` and $c_u \leftrightarrow$ `reward_cap_upper`.

**Figure.** For each penalty cap $c_l$ the solver re-optimises its bid
$s^*(c_l) = \arg\max_s \Pi(s)$. We sweep $c_l$ along the x-axis and track:

- **left panel:** the optimal bid $s^*(c_l)$,
- **right panel:** the **success rate** $P(\text{deliver}\mid\text{win}) = 1 - \text{unset rate}$,
  whose shortfall from $1$ splits into the base rate $p_{\text{fail}}$ and the routing-specific
  (volatility-driven) rate.

Sliders control the surplus ($\mu, \sigma_{\text{std}}$), the base unset rate $p_{\text{fail}}$, and
the reward cap $c_u$; the $c_l$ sweep range auto-scales with the mean surplus.

In [ ]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

In [ ]:
# ----------------------------------------------------------------------
# Model.  sigma is the routing SURPLUS and may be negative (a poor draw
# means delivering at a loss).  With probability p_fail the routing is
# forced to unset; otherwise the solver delivers iff the surplus clears
# its pre-planned threshold sigma_bar(s) and unsets below it.
# ----------------------------------------------------------------------
def gaussian_surplus(mu, sigstd, n, k=4.0):
    """Discretised Gaussian surplus on [mu - k*sigstd, mu + k*sigstd].
    The support spans negative surplus, so a wide draw can come out a loss."""
    edges = np.linspace(mu - k * sigstd, mu + k * sigstd, n + 1)
    mids = 0.5 * (edges[:-1] + edges[1:])
    w = np.exp(-0.5 * ((mids - mu) / sigstd) ** 2)
    return mids, w / w.sum()


def _tail_moments(sig, p):
    """Return tail(tau) -> (P(sigma >= tau), E[sigma * 1{sigma >= tau}]),
    vectorised over an array of thresholds tau, via suffix sums over the
    sorted surplus atoms."""
    order = np.argsort(sig)
    s = sig[order]; pp = p[order]
    suf_p = np.concatenate([np.cumsum(pp[::-1])[::-1], [0.0]])
    suf_s = np.concatenate([np.cumsum((pp * s)[::-1])[::-1], [0.0]])

    def tail(tau):
        idx = np.searchsorted(s, tau, side="left")  # count of atoms < tau
        return suf_p[idx], suf_s[idx]

    return tail


def model_curves(routing, p_fail, S, S_ref, c_u, c_l):
    """Expected payoff Pi(s) and success rate (conditional on winning) over
    the bid grid S.

    Pi(s) = (1/|S_ref|) * sum_{s_ref} 1{win} * E_sigma[payoff]   (losing -> 0).
    With prob p_fail the routing is forced to unset (-P).  Otherwise the
    solver delivers iff sigma >= sigma_bar(s), so the non-forced payoff is
    q R + surplus - q s - (1 - q) P, with q = P(sigma >= sigma_bar(s))."""
    sig, p = routing
    tail = _tail_moments(sig, p)

    Sm = S[:, None]; Sr = S_ref[None, :]
    win = Sm >= Sr                                  # (ns, nr)
    R = np.minimum(c_u, Sm - Sr)                    # reward where win
    P = np.minimum(c_l, Sr)                         # penalty by s_ref
    nwin = win.sum(1)

    # sigma_bar(s) = E[sigma*(s, s_ref) | win], one threshold per bid s.
    sbar = np.where(win, Sm - R - P, 0.0).sum(1) / np.maximum(nwin, 1)

    q, surplus = tail(sbar)                         # deliver prob & delivered surplus (non-forced)
    g_ok = q[:, None] * R + surplus[:, None] - q[:, None] * Sm - (1 - q)[:, None] * P
    g = (1 - p_fail) * g_ok + p_fail * (-P)         # blend non-forced draws with forced unset
    Pi = np.where(win, g, 0.0).sum(1) / len(S_ref)
    success = (1 - p_fail) * q                      # P(deliver | win)
    return dict(Pi=Pi, success=success)


def optima_for_cap(routing, p_fail, S, S_ref, c_u, c_l):
    """Optimal bid s* (argmax expected payoff) and the success rate at s*."""
    cur = model_curves(routing, p_fail, S, S_ref, c_u, c_l)
    i = int(np.argmax(cur["Pi"]))
    return float(S[i]), float(cur["success"][i])


def cap_sweep(routing, p_fail, S, S_ref, c_u, c_l_grid):
    """Optimal bid and success rate for each penalty cap in the grid."""
    out = [optima_for_cap(routing, p_fail, S, S_ref, c_u, c_l) for c_l in c_l_grid]
    s_star, success = (np.array(col) for col in zip(*out))
    return dict(s_star=s_star, success=success)

In [ ]:
# ----------------------------------------------------------------------
# Interactive figure: optimal bid s*(c_l) and success rate at s* as the
# penalty cap c_l sweeps the x-axis.  Sliders set the surplus distribution,
# the base (forced) unset rate p_fail, and the reward cap c_u.
# ----------------------------------------------------------------------
S_REF_MAX = 200.0                          # competing solvers bid s_ref ~ Uniform[0, S_REF_MAX]
S      = np.arange(0, 201, 1.0)            # the solver's own candidate bids s
S_ref  = np.linspace(0.0, S_REF_MAX, 101)  # uniform support for s_ref; averaged with equal weight
N_CAPS = 201                               # resolution of the c_l sweep
N_BINS = 200                               # surplus discretisation bins

plt.ioff()
fig, (axL, axR) = plt.subplots(1, 2, figsize=(11.5, 4.6), constrained_layout=True)
plt.ion()
fig.canvas.header_visible = False

(lineL,) = axL.plot([], [], color="dodgerblue", lw=2, label="optimal bid $s^*$")
mu_line  = axL.axhline(0.0, ls="--", lw=1, color="seagreen",   alpha=0.8)
emu_line = axL.axhline(0.0, ls="--", lw=1, color="darkorange", alpha=0.8)
(lineR,) = axR.plot([], [], color="dodgerblue", lw=2, label="success rate")
ceil_line = axR.axhline(0.0, ls="--", lw=1, color="gray", alpha=0.8)

axL.set(xlabel="penalty cap  $c_l$", ylabel="optimal bid  $s^*$", title="Optimal bid")
axR.set(xlabel="penalty cap  $c_l$", ylabel="success rate   $P(\\mathrm{deliver}\\mid\\mathrm{win})$",
        title="Success rate at $s^*$", ylim=(0, 1.02))

sl = dict(
    mu     = widgets.FloatSlider(value=120, min=0,    max=300, step=1,    description="μ"),
    sigstd = widgets.FloatSlider(value=60,  min=1,    max=120, step=1,    description="σ_std (vol)"),
    p_fail = widgets.FloatSlider(value=0.05,min=0,    max=0.5, step=0.01, description="p_fail (base)",
                                 readout_format=".2f"),
    c_u    = widgets.FloatSlider(value=200, min=0,    max=500, step=5,    description="c_u (reward)"),
)

def update(_=None):
    v = {k: w.value for k, w in sl.items()}
    routing = gaussian_surplus(v["mu"], v["sigstd"], N_BINS)
    c_l_max = max(20.0, 1.1 * (1 - v["p_fail"]) * v["mu"])
    c_l_grid = np.linspace(0, c_l_max, N_CAPS)
    sw = cap_sweep(routing, v["p_fail"], S, S_ref, v["c_u"], c_l_grid)

    lineL.set_data(c_l_grid, sw["s_star"])
    lineR.set_data(c_l_grid, sw["success"])

    top = 1 - v["p_fail"]
    ceil_line.set_ydata([top, top])
    ceil_line.set_label(f"no strategic unset  $1-p_{{fail}}$ = {top:.2f}")

    mu = v["mu"]; emu = (1 - v["p_fail"]) * v["mu"]   # E[surplus | success] and E[surplus]
    mu_line.set_ydata([mu, mu]); mu_line.set_label(f"$\\mu$ = {mu:.0f}")
    emu_line.set_ydata([emu, emu]); emu_line.set_label(f"$(1-p_{{fail}})\\mu$ = {emu:.0f}")

    axL.set_xlim(0, c_l_max); axR.set_xlim(0, c_l_max)
    axL.set_ylim(0, max(1.0, float(sw["s_star"].max()), mu) * 1.08)
    axL.legend(loc="best", fontsize=8)
    axR.legend(loc="best", fontsize=8)
    fig.suptitle(f"μ={v['mu']:.0f}   σ_std={v['sigstd']:.0f}   p_fail(base)={v['p_fail']:.2f}   "
                 f"reward cap c_u={v['c_u']:.0f}", fontsize=9)
    fig.canvas.draw_idle()

for w in sl.values():
    w.observe(update, names="value")
update()

display(widgets.HBox([widgets.VBox(list(sl.values())), fig.canvas]))

### Reading the figure

The right panel decomposes the unset rate top-down:

- the gap from **1.0** to the dashed line **$1 - p_{\text{fail}}$** is the **base** (forced) unset
  rate — the floor the solver can't avoid;
- the gap from the dashed line down to the **success curve** is the **routing-specific** unset rate
  — the strategic unsets driven by volatility.

- **Success rate.** A cheap penalty makes the threshold $\bar\sigma(s)$ high, so the solver unsets
  on more low / negative draws and the success rate sits well below $1 - p_{\text{fail}}$. As $c_l$
  rises the threshold falls and the success rate climbs. It does **not** reach $1 - p_{\text{fail}}$:
  even at a large cap the solver still unsets on negative-surplus draws, because the penalty is
  capped at $s_{\text{ref}}$, so paying out of pocket on a negative draw is never worth it.
- **Volatility ($\sigma_{\text{std}}$).** Raising it widens the surplus distribution, pushes more
  mass below $\bar\sigma(s)$, and shifts the whole success curve **down** — a more volatile token
  unsets more, at every cap.
- **Optimal bid.** A cheaper unset option (small $c_l$) lets the solver bid a little more
  aggressively to win; the bid settles down as the cap rises and the option loses value.
- **Reference lines (left).** The dashed lines mark the expected routing surplus,
  $(1-p_{\text{fail}})\mu$, and its value conditional on the routing not failing, $\mu$; comparing
  $s^*$ against them shows how the bid tracks the surplus the solver expects to capture.